In [ ]:
#%run prelude.rc

import enum
import importlib.util
import sys
from pathlib import Path

import pyarrow

import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob

#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt
import charmingbeauty as cb
lo = cb.layout
cb.visual.set_style_present()



import re
!export DJANGO_ALLOW_ASYNC_UNSAFE=1
import os
from matplotlib import font_manager
from matplotlib import rcParams


os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'
plt.rcParams.update({'text.usetex' : False})


from matplotlib import font_manager


rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Open Sans']



import numpy as np
import polars as pl
import numpy as np
import polars as pl

def average_every_n_by_board(df, n, time_col="timestamp", board_col="board_id"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')
    if board_col not in df.columns:
        raise ValueError(f'{board_col} not in dataframe')

    out = []
    boards = np.unique(df[board_col].to_numpy())
       
    for b in boards:
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )
        
        m = len(sub)
        if m == 0:
            continue

        # make consecutive bins AFTER sorting
        bin_id = np.arange(m) // n
        sub = sub.with_columns(pl.Series("bin_id", bin_id))
        
        exprs = []
        for c, dt in zip(sub.columns, sub.dtypes):
            if c in [board_col, "bin_id"]:
                continue
            if c == time_col:
                exprs.append(pl.col(c).mean().alias(c))
            elif dt.is_numeric():
                exprs.append(pl.col(c).mean().alias(c))
                
        agg = (
            sub.group_by("bin_id", maintain_order=True)
               .agg(exprs)
               .with_columns(pl.lit(b).alias(board_col))
               .drop("bin_id")
               .sort(time_col)
        )
        
        out.append(agg)

    if not out:
        return pl.DataFrame()

    return pl.concat(out).sort([board_col, time_col])





def average_every_n(df, n, time_col="timestamp"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')

    sub = df.sort(time_col)

    m = len(sub)
    bin_id = np.arange(m) // n
    sub = sub.with_columns(pl.Series("bin_id", bin_id))

    exprs = []
    for c, dt in zip(sub.columns, sub.dtypes):
        if c == "bin_id":
            continue
        if c == time_col:
            exprs.append(pl.col(c).mean().alias(c))
        elif dt.is_numeric():
            exprs.append(pl.col(c).mean().alias(c))

    return (
        sub.group_by("bin_id", maintain_order=True)
           .agg(exprs)
           .drop("bin_id")
           .sort(time_col)
    )

from pathlib import Path

def compress_df_by_timebin(df, dt=10.0, time_col="timestamp", board_col="board_id"):
    if len(df) == 0:
        return df

    df = df.sort([board_col, time_col])

    df = df.with_columns(
        (pl.col(time_col) / dt).floor().cast(pl.Int64).alias("tbin")
    )

    key_cols = [board_col, "tbin"]

    exprs = [
        pl.col(time_col).mean().alias(time_col)
    ]

    for c, dtp in zip(df.columns, df.dtypes):
        if c in key_cols or c == time_col:
            continue
        if dtp.is_numeric():
            exprs.append(pl.col(c).mean().alias(c))

    out = (
        df.group_by(key_cols)
          .agg(exprs)
          .drop("tbin")
          .sort([board_col, time_col])
    )

    return out




import polars as pl

def shift_local_time(df, last_t, dt, time_col="timestamp"):
    if len(df) == 0:
        return df, last_t

    first_local = float(df[time_col][0])
    offset = last_t + dt - first_local

    df = df.with_columns(
        (pl.col(time_col) + offset).alias(time_col)
    )

    new_last_t = float(df[time_col][-1])
    return df, new_last_t


import numpy as np
import polars as pl

def estimate_dt_by_board(df, board_col="board_id", time_col="timestamp", min_points=5):
    dt_map = {}

    if len(df) == 0:
        return dt_map

    boards = df[board_col].unique().to_list()

    for b in boards:
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )

        t = sub[time_col].to_numpy()
        if len(t) < min_points:
            continue

        dt = np.diff(t)
        dt = dt[np.isfinite(dt) & (dt > 0)]

        if len(dt) == 0:
            continue

        dt_map[b] = float(np.median(dt))

    return dt_map

def estimate_dt(df, time_col="timestamp", min_points=5):
    if len(df) == 0:
        return None

    t = df.sort(time_col)[time_col].to_numpy()
    if len(t) < min_points:
        return None

    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]

    if len(dt) == 0:
        return None

    return float(np.median(dt))

def shift_local_time_by_board(df, last_t_map, dt_map, board_col="board_id", time_col="timestamp"):
    if len(df) == 0:
        return df, last_t_map

    out = []

    for b in df[board_col].unique().to_list():
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )

        if len(sub) == 0:
            continue

        t0 = float(sub[time_col][0])

        last_t = last_t_map.get(b, None)
        dt = dt_map.get(b, None)

        if dt is None:
            # fallback: estimate from this chunk itself
            t = sub[time_col].to_numpy()
            d = np.diff(t)
            d = d[np.isfinite(d) & (d > 0)]
            dt = float(np.median(d)) if len(d) else 0.0

        if last_t is None:
            offset = -t0
        else:
            offset = last_t + dt - t0

        sub = sub.with_columns(
            (pl.col(time_col) + offset).alias(time_col)
        )

        last_t_map[b] = float(sub[time_col].max())
        out.append(sub)

    if not out:
        return df, last_t_map

    return pl.concat(out).sort([board_col, time_col]), last_t_map




def shift_local_time(df, last_t, dt, time_col="timestamp"):
    if len(df) == 0:
        return df, last_t

    df = df.sort(time_col)

    t0 = float(df[time_col][0])

    if dt is None:
        t = df[time_col].to_numpy()
        d = np.diff(t)
        d = d[np.isfinite(d) & (d > 0)]
        dt = float(np.median(d)) if len(d) else 0.0

    if last_t is None:
        offset = -t0
    else:
        offset = last_t + dt - t0

    df = df.with_columns(
        (pl.col(time_col) + offset).alias(time_col)
    )

    last_t = float(df[time_col].max())
    return df, last_t

import polars as pl
import gondola as gon





PB_to_RB = {
    18: 3,
    2: 32,
    14: 31,
    23: 35,
    3: 23,
    25: 27,
    1: 20,
    4: 16,
    13: 8,
    15: 1,
    5: 26,
    22: 39,
    9: 9,
    7: 41,
    6: 2,
    12: 46,
    21: 7,
    20: 33,
    8: 36,
    11: 28,
}

RB_to_PB = {v: k for k, v in PB_to_RB.items()}





In [ ]:
def build_paddle_map():
    raw = """
1	A   04-11	16
1	B	04-12	16
2	A	04-09	16
2	B	04-10	16
3	A	04-07	16
3	B	04-08	16
4	A	04-05	16
4	B	04-06	16
5	A	04-03	16
5	B	04-04	16
6	A	04-01	16
6	B	04-02	16
7	A	12-16	46
7	B	12-15	46
8	A	12-14	46
8	B	12-13	46
9	A	12-12	46
9	B	12-11	46
10	A	12-10	46
10	B	12-09	46
11	A	12-08	46
11	B	12-07	46
12	A	12-06	46
12	B	12-05	46
13	A	15-02	1
13	B	15-01	1
14	A	15-04	1
14	B	15-03	1
15	A	15-06	1
15	B	15-05	1
16	A	15-08	1
16	B	15-07	1
17	A	15-10	1
17	B	15-09	1
18	A	15-12	1
18	B	15-11	1
19	A	07-05	41
19	B	07-06	41
20	A	07-07	41
20	B	07-08	41
21	A	07-09	41
21	B	07-10	41
22	A	07-11	41
22	B	07-12	41
23	A	07-13	41
23	B	07-14	41
24	A	07-15	41
24	B	07-16	41
25	A	04-14	16
25	B	04-13	16
26	A	04-16	16
26	B	04-15	16
27	A	13-12	8
27	B	13-11	8
28	A	13-10	8
28	B	13-09	8
29	A	13-08	8
29	B	13-07	8
30	A	13-06	8
30	B	13-05	8
31	A	13-04	8
31	B	13-03	8
32	A	13-02	8
32	B	13-01	8
33	A	22-10	39
33	B	22-09	39
34	A	22-08	39
34	B	22-07	39
35	A	22-12	39
35	B	22-11	39
36	A	22-06	39
36	B	22-05	39
37	A	22-14	39
37	B	22-13	39
38	A	22-04	39
38	B	22-03	39
39	A	22-16	39
39	B	22-15	39
40	A	22-02	39
40	B	22-01	39
41	A	12-04	46
41	B	12-03	46
42	A	12-02	46
42	B	12-01	46
43	A	06-06	2
43	B	06-05	2
44	A	06-08	2
44	B	06-07	2
45	A	06-10	2
45	B	06-09	2
46	A	06-12	2
46	B	06-11	2
47	A	06-14	2
47	B	06-13	2
48	A	06-16	2
48	B	06-15	2
49	A	08-10	36
49	B	08-09	36
50	A	08-08	36
50	B	08-07	36
51	A	08-12	36
51	B	08-11	36
52	A	08-06	36
52	B	08-05	36
53	A	08-14	36
53	B	08-13	36
54	A	08-04	36
54	B	08-03	36
55	A	08-16	36
55	B	08-15	36
56	A	08-02	36
56	B	08-01	36
57	A	05-04	26
57	B	05-03	26
58	A	09-14	9
58	B	09-13	9
59	A	21-04	7
59	B	21-03	7
60	A	19-14	20
60	B	19-13	20
61	A	18-11	3
61	B	18-12	3
62	A	18-09	3
62	B	18-10	3
63	A	18-07	3
63	B	18-08	3
64	A	18-05	3
64	B	18-06	3
65	A	18-03	3
65	B	18-04	3
66	A	18-01	3
66	B	18-02	3
67	A	02-02	32
67	B	02-01	32
68	A	02-04	32
68	B	02-03	32
69	A	02-06	32
69	B	02-05	32
70	A	02-08	32
70	B	02-07	32
71	A	02-10	32
71	B	02-09	32
72	A	02-12	32
72	B	02-11	32
73	A	18-13	3
73	B	18-14	3
74	A	18-15	3
74	B	18-16	3
75	A	14-01	31
75	B	14-02	31
76	A	14-03	31
76	B	14-04	31
77	A	14-05	31
77	B	14-06	31
78	A	14-07	31
78	B	14-08	31
79	A	03-15	23
79	B	03-16	23
80	A	03-13	23
80	B	03-14	23
81	A	03-11	23
81	B	03-12	23
82	A	03-09	23
82	B	03-10	23
83	A	03-07	23
83	B	03-08	23
84	A	03-05	23
84	B	03-06	23
85	A	03-03	23
85	B	03-04	23
86	A	03-01	23
86	B	03-02	23
87	A	23-15	35
87	B	23-16	35
88	A	23-13	35
88	B	23-14	35
89	A	23-11	35
89	B	23-12	35
90	A	23-09	35
90	B	23-10	35
91	A	02-14	32
91	B	02-13	32
92	A	02-16	32
92	B	02-15	32
93	A	23-01	35
93	B	23-02	35
94	A	23-03	35
94	B	23-04	35
95	A	23-05	35
95	B	23-06	35
96	A	23-07	35
96	B	23-08	35
97	A	25-15	27
97	B	25-16	27
98	A	25-13	27
98	B	25-14	27
99	A	25-11	27
99	B	25-12	27
100	A	25-09	27
100	B	25-10	27
101	A	25-07	27
101	B	25-08	27
102	A	25-05	27
102	B	25-06	27
103	A	25-03	27
103	B	25-04	27
104	A	25-01	27
104	B	25-02	27
105	A	14-15	31
105	B	14-16	31
106	A	14-13	31
106	B	14-14	31
107	A	14-11	31
107	B	14-12	31
108	A	14-09	31
108	B	14-10	31
109	A	13-16	8
109	B	13-15	8
110	A	13-14	8
110	B	13-13	8
111	A	15-16	1
111	B	15-15	1
112	A	15-14	1
112	B	15-13	1
113	A	05-10	26
113	B	05-09	26
114	A	05-08	26
114	B	05-07	26
115	A	05-06	26
115	B	05-05	26
116	A	19-02	20
116	B	19-01	20
117	A	19-04	20
117	B	19-03	20
118	A	19-06	20
118	B	19-05	20
119	A	11-16	28
119	B	11-15	28
120	A	11-14	28
120	B	11-13	28
121	A	11-12	28
121	B	11-11	28
122	A	11-10	28
122	B	11-09	28
123	A	11-08	28
123	B	11-07	28
124	A	11-06	28
124	B	11-05	28
125	A	11-04	28
125	B	11-03	28
126	A	11-02	28
126	B	11-01	28
127	A	09-16	9
127	B	09-15	9
128	A	05-02	26
128	B	05-01	26
129	A	06-02	2
129	B	06-01	2
130	A	06-04	2
130	B	06-03	2
131	A	07-02	41
131	B	07-01	41
132	A	07-04	41
132	B	07-03	41
133	A	09-08	9
133	B	09-07	9
134	A	09-10	9
134	B	09-09	9
135	A	09-12	9
135	B	09-11	9
136	A	21-16	7
136	B	21-15	7
137	A	21-14	7
137	B	21-13	7
138	A	21-12	7
138	B	21-11	7
139	A	20-02	33
139	B	20-01	33
140	A	20-04	33
140	B	20-03	33
141	A	20-06	33
141	B	20-05	33
142	A	20-08	33
142	B	20-07	33
143	A	20-10	33
143	B	20-09	33
144	A	20-12	33
144	B	20-11	33
145	A	20-14	33
145	B	20-13	33
146	A	20-16	33
146	B	20-15	33
147	A	21-02	7
147	B	21-01	7
148	A	19-16	20
148	B	19-15	20
149	A	05-12	26
149	B	05-11	26
150	A	05-14	26
150	B	05-13	26
151	A	05-16	26
151	B	05-15	26
152	A	09-01	9
152	B	09-02	9
153	A	09-03	9
153	B	09-04	9
154	A	09-05	9
154	B	09-06	9
155	A	21-05	7
155	B	21-06	7
156	A	21-07	7
156	B	21-08	7
157	A	21-09	7
157	B	21-10	7
158	A	19-08	20
158	B	19-07	20
159	A	19-10	20
159	B	19-09	20
160	A	19-12	20
160	B	19-11	20
""".strip().splitlines()

    paddle_map = {}

    for line in raw:
        parts = line.split()
        paddle = int(parts[0])
        side   = parts[1]
        pb_ch  = parts[2]
        rb     = int(parts[3])

        pb, ch = pb_ch.split("-")
        pb = int(pb)
        ch = int(ch)

        signed_id = -paddle if side == "A" else paddle

        paddle_map[signed_id] = {
            "rb": rb,
            "pb": pb,
            "ch": ch
        }

    return paddle_map


paddle_map = build_paddle_map()

def build_pbch_to_paddle_map(paddle_map):
    pbch_to_paddle = {}

    for signed_pid, info in paddle_map.items():
        key = (info["pb"], info["ch"])

        if key in pbch_to_paddle:
            raise ValueError(f"Duplicate mapping for {key}")

        pbch_to_paddle[key] = signed_pid

    return pbch_to_paddle


pbch_to_paddle = build_pbch_to_paddle_map(paddle_map)

In [ ]:
print("gondola version:", gon.__version__)
print("gondola path:", gon.__file__)

db = gon.db


paddles = db.TofPaddle.all()
#paddles.__dir__()

#(DSI,J, channel) -> (paddle_ID, panel_ID)

dsiJ_chP = db.get_dsi_j_ch_pid_map()
for key in dsiJ_chP.keys():
    for key2 in dsiJ_chP[key]:
        for key2 in dsiJ_chP[key]:
            print(key,key2, dsiJ_chP[key][key2])
            print("\n\n")

    
rats = db.RAT.objects.all()

paddle_to_rb_id = {pdl.paddle_id : pdl.rb_id for pdl in paddles}
paddle_to_ltb_dict   = {pdl.paddle_id : pdl.ltb_id for pdl in paddles}
paddle_dict = {pdl.paddle_id : pdl for pdl in paddles}
ltb_dict = {l.board_id : l for l in ltbs}
ltb_to_paddle_dict = {ltb.board_id : [ltb.paddle1_id, ltb.paddle2_id,ltb.paddle3_id,ltb.paddle4_id,ltb.paddle5_id,ltb.paddle6_id,ltb.paddle7_id,ltb.paddle8_id]  for ltb in ltbs}

for i in range(160):
    paddle_id = i + 1
    ltb_board_id = paddle_to_ltb_dict[paddle_id]
    ltb = ltb_dict[ltb_board_id]
    count = 1
    for v in ltb_to_paddle_dict[ltb_board_id]:
        if paddle_id == v:
            channels = (2*count - 1 , 2*count)
            break;
        count = count + 1
    final = (ltb.dsi, ltb.j, channels)


def create_final_dict(paddle_to_ltb_dict, ltb_dict, ltb_to_paddle_dict):
    final_dict = {}
    
    for i in range(160):
        paddle_id = i + 1
        ltb_board_id = paddle_to_ltb_dict[paddle_id]
        ltb = ltb_dict[ltb_board_id]
        
        count = 1
        for v in ltb_to_paddle_dict[ltb_board_id]:
            if paddle_id == v:
                channels = (2 * count - 1, 2 * count)
                break
            count += 1
        
        final = (ltb.dsi, ltb.j, channels)
        
        # Store the result in the dictionary
        final_dict[paddle_id] = final

        # Optionally, you can print the result
        #print(paddle_id)
        #print(final)
    return final_dict




paddle_to_multich = create_final_dict(paddle_to_ltb_dict, ltb_dict, ltb_to_paddle_dict)


#print(paddle_to_multich)

def invert_final_dict(final_dict):
    inverted_dict = {}
    
    for paddle_id, final_value in final_dict.items():
        # Use the `final_value` tuple as the key and `paddle_id` as the value
        inverted_dict[final_value] = paddle_id
    
    return inverted_dict






multich_to_paddle = invert_final_dict(paddle_to_multich)



In [ ]:

%matplotlib inline


plt.rcParams["font.family"] = "DejaVu Sans"
import gondola as gon
import time

import channel_rates


files = gon.io.grace_get_telemetry_binaries(
    1766959800, #start 1765835400
    1767039800, #1766949800
    #, #end time 1767979800 (testing it is for the random 100,000 seconds of flight)
    '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
)


chunk_size = 500

# global accumulated outputs
dfPA = dfCPU = dfRB = dfLTB = dfMTB = None

# running last times
pa_last_t_map = {}
rb_last_t_map = {}
ltb_last_t_map = {}

cpu_last_t = None
mtb_last_t = None

# cadence maps
pa_dt_map = {}
rb_dt_map = {}
ltb_dt_map = {}

cpu_dt = None
mtb_dt = None


lpt = None
toml_find = False


dfSIP = dfPA = dfCPU = dfRB = dfLTB = dfMTB = None

pa_last_t  = -2.0
cpu_last_t = -2.0
rb_last_t  = -10.0
ltb_last_t = -2.0
mtb_last_t = -10.0
sip_last_t = -30
pb_last_t = -2.0


pa_dt  = 2.0
cpu_dt = 2.0
rb_dt  = 10.0
ltb_dt = 2.0
mtb_dt = 10.0
sip_dt = 30
pb_dt = 2


byChProcessor = channel_rates.processor()


fileCount = 0
for i in range(0, len(files), chunk_size):
    print(i)
    print("chunk^")
    chunk = files[i:i+chunk_size]
    
    pa   = gon.monitoring.PAMoniDataSeries()
    cpuM = gon.monitoring.CPUMoniDataSeries()
    rbM  = gon.monitoring.RBMoniDataSeries()
    ltbM = gon.monitoring.LTBMoniDataSeries()
    mtbM = gon.monitoring.MasterTriggerHBSeries()
    sipM = gon.monitoring.SipPosMoniDataSeries()
    pbM  = gon.monitoring.PBMoniDataSeries()
    
    for m in [pa, cpuM, rbM, ltbM, mtbM, sipM]:
        m.max_size = int(1e7)
    
    
    for f in chunk:
        #monitoring information
        sf = str(f)
        pa.add_telemetryfile(sf)
        cpuM.add_telemetryfile(sf)
        rbM.add_telemetryfile(sf)
        ltbM.add_telemetryfile(sf)
        mtbM.add_telemetryfile(sf)
        sipM.add_telemetryfile(sf)
        pbM.add_telemetryfile(sf)
        byChProcessor.process(sf)
        
    
    
    dfPAd_raw  = pa.get_dataframe()
    dfCPUd_raw = cpuM.get_dataframe()
    dfRBd_raw  = rbM.get_dataframe()
    dfLTBd_raw = ltbM.get_dataframe()
    dfMTBd_raw = mtbM.get_dataframe()
    dfSIPd_raw = sipM.get_dataframe()
    dfPBd_raw = pbM.get_dataframe()
    
    
    # learn cadence from first non-empty chunk
    if not pa_dt_map and len(dfPAd_raw):
        pa_dt_map = estimate_dt_by_board(dfPAd_raw, board_col="board_id", time_col="timestamp")
        print("PA dt map:", pa_dt_map)
    
    if not rb_dt_map and len(dfRBd_raw):
        rb_dt_map = estimate_dt_by_board(dfRBd_raw, board_col="board_id", time_col="timestamp")
        print("RB dt map:", rb_dt_map)
    
    if not ltb_dt_map and len(dfLTBd_raw):
        ltb_dt_map = estimate_dt_by_board(dfLTBd_raw, board_col="board_id", time_col="timestamp")
        print("LTB dt map:", ltb_dt_map)
    
    if cpu_dt is None and len(dfCPUd_raw):
        cpu_dt = estimate_dt(dfCPUd_raw, time_col="timestamp")
        print("CPU dt:", cpu_dt)
    
    if mtb_dt is None and len(dfMTBd_raw):
        mtb_dt = estimate_dt(dfMTBd_raw, time_col="total_elapsed")
        print("MTB dt:", mtb_dt)
    
    if sip_dt is None and len(dfSIPd_raw):
        sip_dt = estimate_dt(dfSIPd_raw, time_col="total_elapsed")
        print("SIP dt:", sip_dt)
    
    
    # shift raw chunk times to global
    if len(dfPAd_raw):
        dfPAd_raw, pa_last_t_map = shift_local_time_by_board(
            dfPAd_raw, pa_last_t_map, pa_dt_map,
            board_col="board_id", time_col="timestamp"
        )
    
    if len(dfRBd_raw):
        dfRBd_raw, rb_last_t_map = shift_local_time_by_board(
            dfRBd_raw, rb_last_t_map, rb_dt_map,
            board_col="board_id", time_col="timestamp"
        )
    
    
    if len(dfLTBd_raw):
        dfLTBd_raw, ltb_last_t_map = shift_local_time_by_board(
            dfLTBd_raw, ltb_last_t_map, ltb_dt_map,
            board_col="board_id", time_col="timestamp"
        )

    if len(dfCPUd_raw):
        dfCPUd_raw, cpu_last_t = shift_local_time(
            dfCPUd_raw, cpu_last_t, cpu_dt, time_col="timestamp"
        )

    if len(dfMTBd_raw):
        dfMTBd_raw, mtb_last_t = shift_local_time(
            dfMTBd_raw, mtb_last_t, mtb_dt, time_col="total_elapsed"
        )
    if len(dfSIPd_raw):
        dfSIPd_raw, sip_last_t = shift_local_time(
            dfSIPd_raw, sip_last_t, sip_dt, time_col="timestamp"
        )
    if len(dfPBd_raw):
        dfPBd_raw, pb_last_t = shift_local_time(
            dfPBd_raw, pb_last_t, pb_dt, time_col="timestamp"
        )

    # compress after shifting
    dfPAd  = average_every_n_by_board(dfPAd_raw, 5) if len(dfPAd_raw) else dfPAd_raw
    dfCPUd = average_every_n(dfCPUd_raw, 5, time_col="timestamp") if len(dfCPUd_raw) else dfCPUd_raw
    dfRBd  = average_every_n_by_board(dfRBd_raw, 5) if len(dfRBd_raw) else dfRBd_raw
    dfLTBd = average_every_n_by_board(dfLTBd_raw, 5) if len(dfLTBd_raw) else dfLTBd_raw
    dfMTBd = average_every_n(dfMTBd_raw, 5, time_col="total_elapsed") if len(dfMTBd_raw) else dfMTBd_raw
    dfSIPd = average_every_n(dfSIPd_raw, 5, time_col="timestamp") if len(dfSIPd_raw) else dfSIPd_raw
    
    # accumulate
    dfPA  = dfPAd  if dfPA  is None else pl.concat([dfPA,  dfPAd],  how="vertical_relaxed")
    dfCPU = dfCPUd if dfCPU is None else pl.concat([dfCPU, dfCPUd], how="vertical_relaxed")
    dfRB  = dfRBd  if dfRB  is None else pl.concat([dfRB,  dfRBd],  how="vertical_relaxed")
    dfLTB = dfLTBd if dfLTB is None else pl.concat([dfLTB, dfLTBd], how="vertical_relaxed")
    dfMTB = dfMTBd if dfMTB is None else pl.concat([dfMTB, dfMTBd], how="vertical_relaxed")
    dfSIP = dfSIPd if dfSIP is None else pl.concat([dfSIP, dfSIPd], how="vertical_relaxed")
paddleRateTS, paddleRates = byChProcessor.output_rates()

In [ ]:
print("PA dt map:", pa_dt_map)
print("RB dt map:", rb_dt_map)
print("LTB dt map:", ltb_dt_map)
print("CPU dt:", cpu_dt)
print("MTB dt:", mtb_dt)

#sipM.timestamps looks like 30 seconds


In [ ]:
pa_temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
pa_bias_cols = [c for c in dfPA.columns if c.startswith("biases")]

rb_temp_cols = [c for c in dfRB.columns if c.startswith("tmp_")]
rb_voltage_cols = [c for c in dfRB.columns if c.endswith("_voltage")]
rb_current_cols = [c for c in dfRB.columns if c.endswith("_current")]
rb_power_cols = [c for c in dfRB.columns if c.endswith("_power")]
rb_env_cols = ["pressure", "humidity"]

cpu_temp_cols = [c for c in dfCPU.columns if "temp" in c.lower()]
cpu_freq_cols = [c for c in dfCPU.columns if "freq" in c.lower()]

ltb_reasonable_cols = ["trenz_temp", "ltb_temp", "thresh0", "thresh1", "thresh2"]

mtb_rate_cols = [
    "trate", "lost_trate", "rb_lost_rate", "tiu_busy_rate",
    "trg_lost_trg_rate", "gaps_blocked_rate", "track_blocked_rate",
    "any_blocked_rate", "trkctrl_blocked_rate"
]



def finite_mask(*arrays):
    mask = np.ones(len(arrays[0]), dtype=bool)
    for a in arrays:
        a = np.asarray(a)
        mask &= np.isfinite(a)
    return mask

def range_mask(x, xmin=None, xmax=None):
    x = np.asarray(x)
    mask = np.isfinite(x)
    if xmin is not None:
        mask &= x >= xmin
    if xmax is not None:
        mask &= x <= xmax
    return mask


# PA
PA_TEMP_MIN, PA_TEMP_MAX = -50, 60
PA_BIAS_MIN, PA_BIAS_MAX = 45, 65

# RB
RB_TEMP_MIN, RB_TEMP_MAX = -50, 90
RB_VOLT_MIN, RB_VOLT_MAX = -5, 10
RB_CURR_MIN, RB_CURR_MAX = -1, 10
RB_PWR_MIN,  RB_PWR_MAX  = -1, 50
HUM_MIN, HUM_MAX = 0, 100
PRESS_MIN, PRESS_MAX = 0, 1200

# CPU
CPU_TEMP_MIN, CPU_TEMP_MAX = -20, 120
CPU_FREQ_MIN, CPU_FREQ_MAX = 0, 5000

# LTB
LTB_TEMP_MIN, LTB_TEMP_MAX = -50, 90
THR_MIN, THR_MAX = 0, 1000

# MTB
RATE_MIN, RATE_MAX = 0, 1e6





In [ ]:
# helper
def get_time(df):
    if "timestamp" in df.columns:
        return df["timestamp"].to_numpy()
    if "total_elapsed" in df.columns:
        return df["total_elapsed"].to_numpy()
    raise KeyError(f"No timestamp-like column found. Columns: {df.columns}")


# -------------------------
# 6) RB: voltage sanity
# -------------------------
ax = axes[1, 2]
x = get_time(dfRB)

for c in ["p3v3_voltage", "p3v5_voltage", "zynq_voltage", "drs_dvdd_voltage", "adc_dvdd_voltage"]:
    if c in dfRB.columns:
        y = dfRB[c].to_numpy()
        m = finite_mask(x, y) & range_mask(y, RB_VOLT_MIN, RB_VOLT_MAX)
        ax.plot(x[m], y[m], ".", markersize=2, label=c)

ax.set_xlabel("timestamp")
ax.set_ylabel("voltage (V)")
ax.set_title("RB rail voltages")
ax.legend(fontsize=8)


# -------------------------
# 7) LTB: thresholds
# -------------------------
ax = axes[2, 0]
x = get_time(dfLTB)

for c in ["thresh0", "thresh1", "thresh2"]:
    if c in dfLTB.columns:
        y = dfLTB[c].to_numpy()
        m = finite_mask(x, y) & range_mask(y, THR_MIN, THR_MAX)
        ax.plot(x[m], y[m], ".", markersize=3, label=c)

ax.set_xlabel("timestamp")
ax.set_ylabel("threshold")
ax.set_title("LTB thresholds")
ax.legend(fontsize=8)


# -------------------------
# 8) MTB: trigger losses
# -------------------------
ax = axes[2, 1]
x = get_time(dfMTB)

for c in ["trate", "lost_trate", "rb_lost_rate", "any_blocked_rate"]:
    if c in dfMTB.columns:
        y = dfMTB[c].to_numpy()
        m = finite_mask(x, y) & range_mask(y, RATE_MIN, RATE_MAX)
        ax.plot(x[m], y[m], ".", markersize=3, label=c)

ax.set_xlabel("timestamp")
ax.set_ylabel("rate")
ax.set_title("MTB rates / losses")
ax.legend(fontsize=8)


# -------------------------
# 9) CPU temps
# -------------------------
ax = axes[2, 2]
x = get_time(dfCPU)

for c in cpu_temp_cols:
    if c in dfCPU.columns:
        y = dfCPU[c].to_numpy()
        m = finite_mask(x, y) & range_mask(y, CPU_TEMP_MIN, CPU_TEMP_MAX)
        ax.plot(x[m], y[m], ".", markersize=3, label=c)

ax.set_xlabel("timestamp")
ax.set_ylabel("temperature (C)")
ax.set_title("CPU temperatures")
ax.legend(fontsize=8)

In [ ]:
def corr_subset(df, cols, cuts=None):
    arrs = []
    names = []
    n = len(df)
    mask = np.ones(n, dtype=bool)
    
    for c in cols:
        x = df[c].to_numpy()
        mask &= np.isfinite(x)
        if cuts and c in cuts:
            lo, hi = cuts[c]
            if lo is not None:
                mask &= x >= lo
            if hi is not None:
                mask &= x <= hi
    
    for c in cols:
        arrs.append(df[c].to_numpy()[mask])
        names.append(c)
        
    A = np.column_stack(arrs)
    C = np.corrcoef(A, rowvar=False)
    return names, C
    
def plot_corr(ax, names, C, title):
    im = ax.imshow(C, vmin=-1, vmax=1)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(title)
    return im

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# PA
pa_cols = pa_temp_cols + pa_bias_cols
pa_cuts = {c: (PA_TEMP_MIN, PA_TEMP_MAX) for c in pa_temp_cols}
pa_cuts.update({c: (PA_BIAS_MIN, PA_BIAS_MAX) for c in pa_bias_cols})
names, C = corr_subset(dfPA, pa_cols, pa_cuts)
im = plot_corr(axes[0,0], names, C, "PA: temps and biases")

# RB
rb_cols = ["rate", "pressure", "humidity"] + rb_temp_cols + rb_voltage_cols
rb_cuts = {c: (RB_TEMP_MIN, RB_TEMP_MAX) for c in rb_temp_cols}
rb_cuts.update({c: (RB_VOLT_MIN, RB_VOLT_MAX) for c in rb_voltage_cols})
rb_cuts["pressure"] = (PRESS_MIN, PRESS_MAX)
rb_cuts["humidity"] = (HUM_MIN, HUM_MAX)
rb_cuts["rate"] = (RATE_MIN, RATE_MAX)
names, C = corr_subset(dfRB, rb_cols, rb_cuts)
plot_corr(axes[0,1], names, C, "RB: temps, voltages, env, rate")

# LTB
ltb_cols = ["trenz_temp", "ltb_temp", "thresh0", "thresh1", "thresh2"]
ltb_cuts = {
    "trenz_temp": (LTB_TEMP_MIN, LTB_TEMP_MAX),
    "ltb_temp":   (LTB_TEMP_MIN, LTB_TEMP_MAX),
    "thresh0":    (THR_MIN, THR_MAX),
    "thresh1":    (THR_MIN, THR_MAX),
    "thresh2":    (THR_MIN, THR_MAX),
}
names, C = corr_subset(dfLTB, ltb_cols, ltb_cuts)
plot_corr(axes[1,0], names, C, "LTB: temps and thresholds")

# CPU + MTB reduced
cpu_mtb_cols = ["cpu_temp", "cpu0_temp", "cpu1_temp", "mb_temp"]
cpu_mtb_cols += [c for c in ["trate", "lost_trate", "any_blocked_rate", "tiu_busy_rate"] if c in dfMTB.columns]

# only plot CPU by itself unless you explicitly align timestamps
names, C = corr_subset(dfCPU, [c for c in cpu_temp_cols if c in ["cpu_temp", "cpu0_temp", "cpu1_temp", "mb_temp"]],
                       {c: (CPU_TEMP_MIN, CPU_TEMP_MAX) for c in cpu_temp_cols})
plot_corr(axes[1,1], names, C, "CPU temperatures")

plt.tight_layout()
plt.show()

In [ ]:
def nearest_match(x_ref, x_other, max_dt=30):
    x_ref = np.asarray(x_ref)
    x_other = np.asarray(x_other)

    if len(x_other) == 0:
        raise ValueError("nearest_match: reference comparison array is empty")

    if len(x_other) == 1:
        idx = np.zeros(len(x_ref), dtype=int)
        dt = np.abs(x_other[0] - x_ref)
    else:
        idx = np.searchsorted(x_other, x_ref)
        idx = np.clip(idx, 1, len(x_other) - 1)

        left = idx - 1
        right = idx

        choose_right = np.abs(x_other[right] - x_ref) < np.abs(x_other[left] - x_ref)
        idx = np.where(choose_right, right, left)

        dt = np.abs(x_other[idx] - x_ref)

    # apply max time cut
    if max_dt is not None:
        idx = np.where(dt <= max_dt, idx, -1)

    return idx, dt
    


def getTempAv(dfPA):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]

    # stack into matrix: shape (n_rows, 16)
    temps = np.column_stack([dfPA[c].to_numpy() for c in temp_cols])

    # ignore NaNs automatically
    return np.nanmean(temps, axis=1)

    
def getTempAvB(dfPA, boardID):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .group_by("timestamp")
        .agg([
            *[pl.col(c).mean().alias(c) for c in temp_cols]
        ])
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    temps = np.column_stack([sub[c].to_numpy() for c in temp_cols])
    t = sub["timestamp"].to_numpy()
    avg = np.nanmean(temps, axis=1)

    return t, avg

def match_by_board(dfPA, dfRB):
    # pulls arrays to numpy
    pa_board = dfPA["board_id"].to_numpy()
    rb_board = dfRB["board_id"].to_numpy()
    t_pa = dfPA["timestamp"].to_numpy()
    t_rb = dfRB["timestamp"].to_numpy()

    # makes a big array with all the indexs -1 just incase a match is not found, what is a "match"?
    idx_out = np.full(len(dfPA), -1, dtype=int)

    # loop over unique boards present in PA
    for b in np.unique(pa_board):
        #the mask takes a board at a time
        pa_mask = (pa_board == b)
        rb_mask = (rb_board == b)

        #skips all the rb1s
        if np.sum(rb_mask) == 0:
            continue  # no matching RB for this board
        
        t_pa_sub = t_pa[pa_mask]
        t_rb_sub = t_rb[rb_mask]
        
        #takes the timestamp arrays of the same boards
        
        idx_sub = nearest_match(t_pa_sub, t_rb_sub)
        
        # map back to full indices
        rb_indices = np.where(rb_mask)[0]
        idx_out[pa_mask] = rb_indices[idx_sub]
    return idx_out


def getTempByBoardAndChannel(dfPA, boardID, channelID):
    col = f"temps{channelID}"

    if col not in dfPA.columns:
        raise ValueError(f"Column {col} not found in dfPA")

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .group_by("timestamp")
        .agg(
            pl.col(col).mean().alias(col)
        )
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    t = sub["timestamp"].to_numpy()
    temp = sub[col].to_numpy()

    return t, temp  

def secToHours(sec):
    return sec/3600

In [ ]:
print(dfPA.columns)

In [ ]:
boardID = 7
t, temp_avg = getTempAvB(dfPA, boardID)

plt.figure()
plt.plot(t - t[0], temp_avg, ".")
plt.title(f"Average PA temperature, board {boardID}")
plt.xlabel("Time since start (s)")
plt.ylabel("Temp (C)")
plt.show()

In [ ]:

#now using the mapping...

pbch_to_paddle = build_pbch_to_paddle_map(paddle_map)
for key in paddle_map:
    boardID = paddle_map[key]['rb']
    channel = paddle_map[key]['ch']
    t, temp_avg = getTempByBoardAndChannel(dfPA, boardID, channel)
    if key > 0:
        sideStr = "B";
    else:
        sideStr = "A";
    plt.figure()
    plt.plot(secToHours(t - t[0]), temp_avg, ".")
    plt.title(f"Average PA temperature, paddle {abs(key)} side {sideStr} ")
    plt.xlabel("Time since start (hr)")
    plt.ylabel("Temp (C)")
    plt.show()
    time.sleep(.1)

    
    #print(paddle_map)
    
    
#for key in pbch_to_paddle.keys():

#        print(key)

#        print(pbch_to_paddle[key])

#first create map to do temp to rate for all boards. find correction...

#look, at  threshold studies...

#slopes will be higher for paddles in the inner tof -> altitude change really does it...

In [ ]:
def getTempAvB_raw(dfPA, boardID):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    temps = np.column_stack([sub[c].to_numpy() for c in temp_cols])
    t = sub["timestamp"].to_numpy()
    avg = np.nanmean(temps, axis=1)

    return t, avg


boardID = 7
t, temp_avg = getTempAvB_raw(dfPA, boardID)

plt.figure(figsize=(8,4))
plt.plot(t - t[0], temp_avg, ".")
plt.title(f"Raw average PA temperature, board {boardID}")
plt.xlabel("Time since start (s)")
plt.ylabel("Temp (C)")
plt.show()
    

In [ ]:

print(dfRB.columns)

import numpy as np

def getTempAv(dfPA):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
    temps = np.column_stack([dfPA[c].to_numpy() for c in temp_cols])
    return np.nanmean(temps, axis=1)

temp_avg = getTempAv(dfPA)

idx = match_by_board(dfPA, dfRB)
valid = idx >= 0

print("len(dfPA)    =", len(dfPA))
print("len(temp_avg)=", len(temp_avg))
print("len(idx)     =", len(idx))
print("len(valid)   =", len(valid))



x = temp_avg[valid]
y = dfRB["rate"].to_numpy()[idx[valid]]

plt.figure(figsize=(6,5))
plt.scatter(x, y, s=10, alpha=0.5)

plt.xlabel("Avg PA temp")
plt.ylabel("RB rate")
plt.title("PA temp vs RB rate (same board, time-matched)")

plt.show()




idx = match_by_board(dfPA, dfRB)

valid = idx >= 0  # only rows where match exists



for b in np.unique(dfPA["board_id"].to_numpy()):

    mask = (dfPA["board_id"].to_numpy() == b) & valid

    if np.sum(mask) < 10:
        continue

    x = temp_avg[mask]
    y = dfRB["rate"].to_numpy()[idx[mask]]

    plt.figure(figsize=(5,4))
    plt.scatter(x, y, s=10, alpha=0.5)
    plt.xlabel("Temp (C)")
    plt.ylabel("Rate (Hz)")
    plt.title(f"Board {b}")
    plt.show()

In [ ]:
boardID = 7

sub = (
    dfPA
    .filter(pl.col("board_id") == boardID)
    .sort("timestamp")
)

print(
    sub.group_by("timestamp")
       .len()
       .sort("timestamp")
       .head(30)
)
print(dfPA)

In [ ]:


temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
boards = np.unique(dfPA["board_id"].to_numpy())

for b in boards:
    mask = (dfPA["board_id"].to_numpy() == b)

    if np.sum(mask) < 5:
        continue

    t = dfPA["timestamp"].to_numpy()[mask]

    temps = np.column_stack([
        dfPA[c].to_numpy()[mask] for c in temp_cols
    ])

    avg = np.nanmean(temps, axis=1)
    std = np.nanstd(temps, axis=1)

    plt.figure(figsize=(8,5))

    # individual channels
    for i in range(temps.shape[1]):
        plt.scatter(t, temps[:, i], alpha=0.3)

    # average
    plt.plot(t, avg, linewidth=2, label="avg")

    # spread band
    plt.fill_between(t, avg-std, avg+std, alpha=0.2)

    plt.xlabel("Time")
    plt.ylabel("Temp (C)")
    plt.title(f"Board {b} temps (with avg + spread)")
    plt.legend()
    plt.show()

    

In [ ]:
temp_avg = getTempAv(dfPA)

idx = match_by_board(dfPA, dfRB)
valid = idx >= 0

pa_board = dfPA["board_id"].to_numpy()
t_pa = dfPA["timestamp"].to_numpy()
t_rb = dfRB["timestamp"].to_numpy()
rb_rate = dfRB["rate"].to_numpy()



import matplotlib.pyplot as plt
import numpy as np

for b in np.unique(pa_board):

    mask = (pa_board == b) & valid

    if np.sum(mask) < 20:
        continue

    t = t_pa[mask]
    temp = temp_avg[mask]
    rate = rb_rate[idx[mask]]

    fig, axes = plt.subplots(2, 1, figsize=(8,6), sharex=True)

    # -----------------------
    # Temperature vs time
    # -----------------------
    axes[0].plot(t, temp, ".", markersize=3)
    axes[0].set_ylabel("Temp (C)")
    axes[0].set_title(f"Board {int(b)}")

    # -----------------------
    # Rate vs time
    # -----------------------
    axes[1].plot(t, rate, ".", markersize=3)
    axes[1].set_ylabel("Rate")
    axes[1].set_xlabel("Time")

    plt.tight_layout()
    plt.show()
    

In [ ]:
# LTB voltage sensors: one figure per rail, every board overlaid vs time
# (expects dfLTB, pl, np, plt, and get_time from earlier cells)

if dfLTB is None or len(dfLTB) == 0:
    print("dfLTB is empty; skip LTB voltage overlay plots.")
else:
    board_col = "board_id"
    volt_cols = sorted(
        c for c in dfLTB.columns if str(c).endswith("_v") and c not in (board_col,)
    )
    if not volt_cols:
        print("No *_v columns found on dfLTB.")
    else:
        boards = np.sort(np.unique(dfLTB[board_col].to_numpy()))

        for col in volt_cols:
            plt.figure(figsize=(9, 4))
            for b in boards:
                sub = dfLTB.filter(pl.col(board_col) == b).sort("timestamp")
                if len(sub) == 0:
                    continue
                t = get_time(sub)
                y = sub[col].to_numpy()
                plt.plot(t, y, ".-", markersize=2, lw=0.6, label=f"board {int(b)}")
            plt.xlabel("time (same axis as get_time(dfLTB) elsewhere)")
            plt.ylabel(f"{col} (V)")
            plt.title(f"LTB {col} — all boards")
            plt.legend(loc="best", fontsize=7, ncol=2)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
